# 01 — Exploratory Data Analysis (Pima Indians Diabetes Dataset)

**Author:** Rudolph Otoo  
**Date:** 2026  
**Dataset:** Pima Indians Diabetes Database (UCI / Kaggle mirror)  

---

## Objective

Characterise the distribution and correlation structure of the nine clinical
predictor variables and the binary diabetes outcome label. The insights
produced here inform downstream feature-engineering decisions (e.g., imputation
of physiologically impossible zeros) documented in `src/data.py`.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import Paths, Settings
from src.data import process_data, FEATURE_COLUMNS, TARGET_COLUMN
from src.visualize import (
    set_global_style,
    plot_target_distribution,
    plot_feature_distributions,
    plot_correlation_heatmap,
)

set_global_style()

## 1. Data Acquisition & Preprocessing

The raw Pima dataset is fetched once from its public mirror and cached locally.
Physiologically impossible zero values (e.g., `BloodPressure = 0`) are
re-encoded as `NaN` and imputed with the median. The full recipe is documented
in `src/data.py:process_data()`.

In [ ]:
paths = Paths()
frame = process_data(paths)
print(f"Frame shape: {frame.shape}")
frame.head()

## 2. Class Distribution

A critical first check: if the classes are heavily imbalanced, a naïve
``DummyClassifier`` predicting the majority class could achieve ~65 % accuracy
with zero clinical value. This motivates our use of ROC-AUC and F1-score as
primary evaluation metrics.

In [ ]:
y = frame[TARGET_COLUMN]
print(f"Class 0 (no diabetes): {(y == 0).sum()}")
print(f"Class 1 (diabetes):    {(y == 1).sum()}")
print(f"Prevalence of diabetes: {y.mean():.3f}")

plot_target_distribution(y)

## 3. Feature Distributions

Inspect marginal distributions of each predictor. Strong right-skew or
multimodality may warrant non-linear models or log-transforms.

In [ ]:
X = frame[FEATURE_COLUMNS]
X.describe()

In [ ]:
plot_feature_distributions(X)

## 4. Correlation Structure

Spearman correlation is used rather than Pearson to guard against monotonic
but non-linear associations. Strong target correlations suggest the most
predictive features *a priori*.

In [ ]:
plot_correlation_heatmap(frame)

In [ ]:
corr_with_target = frame.corr(method="spearman")[TARGET_COLUMN].drop(TARGET_COLUMN)
print("Spearman correlations with Outcome (ranked):")
print(corr_with_target.sort_values(ascending=False).to_string())

## 5. Summary of EDA Findings

| Finding | Implication |
|---|---|
| Class imbalance (~35 % positive) | Use ROC-AUC / F1; accuracy alone is misleading |
| Glucose and BMI most correlated with Outcome | Expected; validated by clinical literature |
| Right-skewed Insulin distribution | Median imputation preferred over mean |
| `SkinThickness` and `Insulin` have many zeros | Zeros are physiologically impossible; must be imputed |
| No strong inter-feature multicollinearity | Linear models (LogisticRegression) are valid baselines |